# GenProblems

This notebook shows `GenLAProblems` as a problem generator rather than a rendering or solution-workflow package. Each section builds a typical linear algebra exercise, prints the generated matrices with `l_show()`, and highlights what kind of classroom task that generator is meant to support.

In [ ]:
using GenLAProblems, LAlatex, LinearAlgebra, LaTeXStrings, Random
Random.seed!(42)


## Choosing a generator

- Use `gen_gj_pb` when you want a linear system together with a consistent exact solution matrix `X` and right-hand side `B`. This is the right starting point for row-reduction, pivot-column, rank, and free-variable exercises.
- Use `gen_inv_pb`, `gen_lu_pb`, `gen_plu_pb`, and `gen_ldlt_pb` for matrix-factorization exercises. These generators are better than ad hoc random matrices when you want the factorization to come out cleanly and exactly.
- Use `gen_qr_problem(...; family=...)` for QR exercises. The `family` choice controls the style of the hidden orthogonal seed, which in turn affects whether the arithmetic feels sparse, dense, highly structured, or especially hand-computation-friendly.
- Use `gen_eigenproblem` for general diagonalization exercises and `gen_symmetric_eigenproblem` when you specifically want orthogonal diagonalization.
- Use `gen_svd_problem(...; left_family=..., right_family=...)` when you want explicit control over the left and right orthogonal factors in an SVD exercise rather than accepting a single default style.

## Exact arithmetic

These generators are aimed at classroom-friendly exact arithmetic. In most examples you should expect integers or exact rationals rather than floating-point approximations. That is deliberate: the goal is to make the algebra inspectable and reproducible on paper, not to simulate floating-point numerics. Some orthogonal-factor families introduce rational entries, but they still do so in an exact way.

`GenLAProblems` is the problem-generation layer of this stack. It constructs exact linear algebra exercises and supporting matrices. Matrix display here uses `LAlatex`. Algorithm and workflow visualizations are implemented by `LAFigureSpecs` and `matrixlayout`, and exposed in Julia through `LATeachingSuite`.

## Gauss-Jordan problem

Exercise intent: solve for `X`, identify pivot columns and free variables, and interpret the relationship between the coefficient matrix `A`, the solution matrix `X`, and the right-hand side `B`.

Return values: `A, X, B` where `A X = B`.

Common variations: decrease `maxint` for easier arithmetic, increase `num_rhs` to generate multiple right-hand sides at once, or lower the rank parameter `r` to produce more free-variable structure.

In [ ]:
A_ge, X_ge, B_ge = gen_gj_pb(3, 4, 3; maxint=3, num_rhs=2)
l_show(L"A X = B : \qquad ", A_ge, X_ge, " = ", B_ge)


## Inverse problem

Exercise intent: verify that the displayed inverse is correct, check the product explicitly, and use the example as a model for invertibility questions over exact arithmetic.

Return values: `A, A_inv` where `A_inv` is the exact inverse of `A`.

In [ ]:
A, A_inv = gen_inv_pb(3; maxint=2)
l_show(L"A = ", A, L",\quad A^{-1} = ", A_inv, L",\qquad \text{ check product }\quad ", A*A_inv)


## LU problem

Exercise intent: verify `A = LU`, inspect how the unit-lower and upper-triangular factors were chosen, and use the factorization as the natural starting point for forward/back substitution.

Return values: `pivot_cols, L, U, A` with `A = LU`.

Common variations: change the target rank, allow zeros to appear more often, or compare this no-pivoting family directly with the PLU family below when you want row swaps to become part of the exercise.

In [ ]:
pivot_cols_lu, L_lu, U_lu, A_lu = gen_lu_pb(3, 3, 3; maxint=2)
l_show(L"A = LU : \qquad ", A_lu, " = ", L_lu, U_lu)


## PLU problem

Exercise intent: determine why pivoting is needed, verify the `A = PLU` factorization, and compare the structure of this example with the simpler no-pivoting LU case above.

Return values: `pivot_cols, P, L, U, A` with `A = P L U`.

Common variations: change the rank or zero pattern, compare against `gen_lu_pb(...)` to isolate the effect of pivoting, or use a wider matrix when you want factorization ideas to interact with rank questions.

In [ ]:
pivot_cols_plu, P_plu, L_plu, U_plu, A_plu = gen_plu_pb(3, 3, 3; maxint=2)
l_show(L"A = P L U, : \qquad  ", A_plu, " = ", P_plu, L_plu, U_plu)


## QR problem

Exercise intent: compute a QR factorization and compare how the family choice affects the arithmetic. In practical terms:
- `:pythagorean` gives the smallest hand-computation-friendly exact examples.
- `:hadamard` gives highly structured QR seeds when the size supports that family.
- `:cayley` gives denser exact rational orthogonal structure.
- `:sparse` gives block-structured orthogonal factors that are easier to decompose and reason about locally.

Return value: `A`, a matrix meant to be factored as `Q R`.

Common variations: switch the `family` to change the style of the orthogonal seed, decrease `maxint` for simpler arithmetic, or use block sizes such as `(2, 2)` with `family=:sparse` when you want more visible internal structure.

In [ ]:
A_qr_pyth = gen_qr_problem(3; family=:pythagorean, maxint=2)
A_qr_had  = gen_qr_problem(4; family=:hadamard, maxint=2)
A_qr_cay  = gen_qr_problem(5; family=:cayley, maxint=2)
A_qr_sp   = gen_qr_problem((2, 2); family=:sparse, maxint=2)
l_show(L"A_{\mathrm{pyth}} = ", A_qr_pyth, L", \qquad A_{\mathrm{had}} = ", A_qr_had)
l_show(L"A_{\mathrm{cay}} = ", A_qr_cay, L", \qquad A_{\mathrm{sparse}(2,2)} = ", A_qr_sp)


## Eigenvalue problem

Exercise intent: diagonalize `A`, verify the similarity decomposition `A = S \Lambda S^{-1}`, and connect the chosen eigenvalues to the resulting diagonal matrix and change-of-basis matrix.

Return values: `S, \Lambda, S_inv, A` with `A = S \Lambda S^{-1}`.

In [ ]:
S_eig, Lambda_eig, S_inv_eig, A_eig = gen_eigenproblem([3, -1, 2]; maxint=2)
l_show(L"A = S \Lambda S^{-1} :  \qquad ", A_eig, " = ", S_eig, Lambda_eig, S_inv_eig)


## Symmetric eigenvalue problem

Exercise intent: orthogonally diagonalize `A` and verify `A = Q \Lambda Q^T`. This is the symmetric-matrix version of the previous example, so the main point is that the change-of-basis matrix is orthogonal rather than merely invertible.

Return values: `Q, \Lambda, A` with `A = Q \Lambda Q^T`.

In [ ]:
Q_sym, Lambda_sym, A_sym = gen_symmetric_eigenproblem([4, 1, -2]; maxint=2)
l_show(L"A = Q \Lambda Q^T :  \qquad ", A_sym, " = ", Q_sym, Lambda_sym, transpose(Q_sym))


## SVD problem

Exercise intent: verify the SVD and compare how left/right family choices change the orthogonal factors. In practical terms, `left_family` and `right_family` let you decide whether the singular-vector factors should look sparse, highly structured, or more densely rational.

Return values: `U, \Sigma, Vt, A` with `A = U \Sigma V^T`.

Common variations: change `left_family` and `right_family` independently, switch between scalar dimensions and block partitions, or adjust the singular values `\sigma` to control rank and the visual shape of `\Sigma`.

In [ ]:
U_svd, Sigma_svd, Vt_svd, A_svd = gen_svd_problem([2, 1], [2, 1], [3, 1, 0]; maxint=2)
l_show(L"A = U \Sigma V^T : \qquad ", A_svd, " = ", U_svd, Sigma_svd, Vt_svd)


In [ ]:
U_mix, Sigma_mix, Vt_mix, A_mix = gen_svd_problem(4, 4, [3, 1, 0, 0]; left_family=:hadamard, right_family=:cayley, maxint=2)
l_show(L"A = U \Sigma V^T : \qquad ", A_mix, " = ", U_mix, Sigma_mix, Vt_mix)


## Recommended defaults

- GE/GJ systems: `gen_gj_pb(m, n, r; maxint=3, num_rhs=1)` is the best all-purpose default.
- LU vs PLU: use `gen_lu_pb(...)` when you want a no-pivoting exercise and `gen_plu_pb(...)` when row swaps should be part of the story.
- QR: `gen_qr_problem(n; family=:auto, maxint=2)` is the best default unless you already know you want a specific family.
- Symmetric eigenvalue problems: `gen_symmetric_eigenproblem(vals; maxint=2)` is the clearest entrypoint for orthogonal diagonalization exercises.
- SVD: `gen_svd_problem([2, 1], [2, 1], σ; maxint=2)` is a good sparse/block-style exact default when you want a compact, interpretable example.

## Parameter tuning

These examples show how small parameter changes affect the style of the generated exercise. The point is not just that the numbers change: the whole character of the exercise changes, including how dense the matrices are, how many right-hand sides appear, and how structured the orthogonal factors feel.

In [ ]:
A_ge_easy, X_ge_easy, B_ge_easy = gen_gj_pb(3, 4, 3; maxint=1, num_rhs=1)
A_ge_rich, X_ge_rich, B_ge_rich = gen_gj_pb(3, 4, 3; maxint=3, num_rhs=2)
l_show(L"Easy GE: ", A_ge_easy, X_ge_easy, " = ", B_ge_easy)
l_show(L"Richer GE: ", A_ge_rich, X_ge_rich, " = ", B_ge_rich)


In [ ]:
A_qr_auto = gen_qr_problem(4; family=:auto, maxint=1)
A_qr_had = gen_qr_problem(4; family=:hadamard, maxint=3)
l_show(L"QR auto/default style: ", A_qr_auto, L", \qquad QR hadamard style: ", A_qr_had)


In [ ]:
U_sp, Sigma_sp, Vt_sp, A_sp = gen_svd_problem([2, 1], [2, 1], [3, 1, 0]; maxint=2)
U_mix2, Sigma_mix2, Vt_mix2, A_mix2 = gen_svd_problem(4, 4, [3, 1, 0, 0]; left_family=:sparse, right_family=:cayley, maxint=2)
l_show(L"Sparse/block SVD: ", A_sp, L", \qquad Mixed-family SVD: ", A_mix2)
